<a href="https://colab.research.google.com/github/ACH-S-M/Nlp/blob/main/Salinan_dari_PERT_9_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Dataset**

In [ ]:
# ------------------------------------------------------------------
# Download "Tokopedia Product Reviews" dari Kaggle
# dan simpan CSV-nya di folder yang SAMA dengan notebook ini
# ------------------------------------------------------------------

# --- 2. Mount Google Drive (jika belum) ---
import os
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive/')

# --- 3. Tentukan TARGET FOLDER (folder notebook Anda) ---
# ⚠️ Sesuaikan path di bawah dengan folder tempat notebook ini disimpan
TARGET_DIR = '/content/drive/MyDrive/colab'   # <-- UBAH INI

os.makedirs(TARGET_DIR, exist_ok=True)
print("📁 Target folder:", TARGET_DIR)

# --- 4. Download via kagglehub (masuk ke cache dulu) ---
import kagglehub
dataset_cache_path = kagglehub.dataset_download("farhan999/tokopedia-product-reviews")
print("📦 Dataset cached at:", dataset_cache_path)

# --- 5. Copy SEMUA file dari cache ke TARGET_DIR ---
import shutil

copied_files = []
for root, _, files in os.walk(dataset_cache_path):
    for f in files:
        src = os.path.join(root, f)
        dst = os.path.join(TARGET_DIR, f)
        # Hindari overwrite jika file sudah ada
        if os.path.exists(dst):
            print(f"⏭️  Skip (sudah ada): {f}")
        else:
            copied_files.append(f)
            print(f"✅ Copied: {f} -> {dst}")

print(f"\n🎉 {len(copied_files)} file disalin ke {TARGET_DIR}")

# --- 6. Load CSV dari TARGET_DIR ---
import pandas as pd

csv_file = None
for f in os.listdir(TARGET_DIR):
    if f.lower().endswith('2019.csv'):
        csv_file = os.path.join(TARGET_DIR, f)
        break

if csv_file:
    df = pd.read_csv(csv_file)
    print(f"\n✅ Loaded: {csv_file}")
    print(f"📊 Shape: {df.shape}")
    display(df.head())
else:
    print("❌ Tidak ada file CSV di folder target.")

📁 Target folder: /content/drive/MyDrive/colab
Using Colab cache for faster access to the 'tokopedia-product-reviews' dataset.
📦 Dataset cached at: /kaggle/input/tokopedia-product-reviews
⏭️  Skip (sudah ada): tokopedia-product-reviews-2019.csv

🎉 0 file disalin ke /content/drive/MyDrive/colab

✅ Loaded: /content/drive/MyDrive/colab/tokopedia-product-reviews-2019.csv
📊 Shape: (40607, 9)


,Unnamed: 0,text,rating,category,product_name,product_id,sold,shop_id,product_url
0,1,Barang sesuai pesanan dan cepat sampai,5,pertukangan,Staples Dekorasi Staples Kayu + Refill 8mm - S...,418660637,1,1740837,https://www.tokopedia.com/shakaonline87/staple...
1,2,Barang bagus harga murah,5,pertukangan,STAPLE GUN ATS 3 WAY TACKER - STAPLES JOK TEMB...,416032545,11,1477109,https://www.tokopedia.com/juraganperkakas/stap...
2,3,Paket rapi...mantap....cepat....sampe ke tujuan,5,pertukangan,STAPLE GUN ATS 3 WAY TACKER - STAPLES JOK TEMB...,416032545,11,1477109,https://www.tokopedia.com/juraganperkakas/stap...
3,4,ya saya puas dgn barangnya,5,pertukangan,ALAT STAPLES TEMBAK &#40;AIR NAILER GUN&#41; O...,102279869,5,771395,https://www.tokopedia.com/kamarmesin/alat-stap...
4,5,Responya luar biasa b mantap,5,pertukangan,Isi Refill Staples Jok Kulit Motor / Staple Gu...,190679689,787,969999,https://www.tokopedia.com/mitrapersada/isi-ref...




# **Preprocessing dengan Sastrawi**



In [ ]:
# Install library
!pip install Sastrawi nltk

import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.StopWordRemover.StopWordRemover import StopWordRemover


nltk.download('punkt_tab')

# Sastrawi
stemmer = StemmerFactory().create_stemmer()

# Stopword bawaan Sastrawi
factory = StopWordRemoverFactory()
stopwords = factory.get_stop_words()

# Custom kata
custom_stopwords = ['terima', 'kasih', 'makasih', 'gan', 'mantap']

# Gabungkan
stopwords.extend(custom_stopwords)

# Hilangkan duplikasi
stopwords = list(set(stopwords))

# Buat remover
stopword_remover = StopWordRemover(stopwords)

# Penggunaan
teks = "terima kasih gan, produk ini sangat mantap"
hasil = stopword_remover.remove(teks)

print(hasil)
# slang =

# def ubah_slang(kata):
#     if kata in slang:
#         return slang[kata]
#     return kata

def preprocess(text):
    if not isinstance(text, str):
        return ''
    # 1. Case folding
    text = text.lower()
    # 2. Cleaning
    text = re.sub(r'[^a-z\s]', ' ', text)
    # 3. Tokenization
    tokens = word_tokenize(text)
    # 4. Stopword removal
    tokens = [w for w in tokens if w not in stop_words and len(w) > 2]
    # 5. Stemming
    tokens = [stemmer.stem(w) for w in tokens]
    return ' '.join(tokens)

# Test
review = "Barang BAGUS bgt!!! Cepet sampai, packing aman."
print(preprocess(review))
# Output: "barang bagus cepat sampai packing aman"

kasih gan, produk sangat
barang bagus bgt cepet packing aman


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# **TF-IDF pada Review Tokopedia**

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Load data
df = pd.read_csv('/content/drive/MyDrive/colab/tokopedia-product-reviews-2019.csv')
df.head()
df = df.dropna(subset=['text'])
df['clean'] = df['text'].apply(preprocess)

# TF-IDF
tfidf = TfidfVectorizer(max_features=1000, min_df=5, max_df=0.8)
X = tfidf.fit_transform(df['clean'])

print(f"Shape: {X.shape}")
# (40607 dokumen × 1000 fitur)

# Top 20 kata
import numpy as np
features = tfidf.get_feature_names_out()
avg = np.asarray(X.mean(axis=0)).flatten()
top = avg.argsort()[-20:][::-1]
print([features[i] for i in top])

Shape: (40607, 1000)
['barang', 'sesuai', 'cepat', 'terima', 'bagus', 'kirim', 'baik', 'pesan', 'kasih', 'gan', 'respon', 'mantap', 'deskripsi', 'produk', 'thanks', 'seller', 'terimakasih', 'packing', 'kualitas', 'harga']


# **Sentimen dengan Naive Bayes**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

# Map rating to sentiment (e.g., ratings 4-5 are positive, 1-3 are negative/neutral)
# Adjust this mapping function if your classification criteria are different
df['sentiment'] = df['rating'].apply(lambda x: 'positive' if x >= 4 else ('neutral' if x == 3 else 'negative'))

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, df['sentiment'], test_size=0.2, random_state=42,
    stratify=df['sentiment']
)

# Model
model = MultinomialNB()
model.fit(X_train, y_train)

# Prediksi
y_pred = model.predict(X_test)

# Evaluasi
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    negative       0.61      0.15      0.24       185
     neutral       0.21      0.03      0.05       365
    positive       0.94      0.99      0.96      7572

    accuracy                           0.93      8122
   macro avg       0.58      0.39      0.42      8122
weighted avg       0.90      0.93      0.91      8122



# **Topik Modelling: LDA (Latent Dirichlet Allocation)**

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(X)

features = tfidf.get_feature_names_out()
for i, topic in enumerate(lda.components_):
    top = [features[j] for j in topic.argsort()[-10:][::-1]]
    print(f"Topik {i+1}: {' | '.join(top)}")

Topik 1: good | nya | bagus | lama | pakai | barang | kirim | kurang | langsung | hari
Topik 2: makasih | gan | terimakasih | barang | bintang | udah | sampe | warna | dah | bicara
Topik 3: terima | barang | pesan | kasih | baik | sesuai | cepat | thanks | fungsi | fast
Topik 4: mantap | sukses | barang | selamat | cepet | top | gan | banget | thank | selalu
Topik 5: sesuai | cepat | bagus | barang | kirim | deskripsi | produk | harga | kualitas | respon
